# Predicting Corporate Bankruptcy from Financial Statements

**Author:** Mauro Reverberi

**Program:** MSc AI, Udacity Institute of AI & Technology / Woolf

**Project:** Capstone Project 3, Machine Learning Foundations

**Dataset:** Polish Companies Bankruptcy Data, UCI Machine Learning Repository (dataset 365), license CC BY 4.0:
https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data

In this notebook I train a model that predicts from one year of financial
indicators whether a company will go bankrupt within the next three
years. The input is 64 financial indicators from a published annual statement.
The output is a bankruptcy risk score.

**Research question:** How well can supervised machine learning predict, from
one year of financial indicators, whether a company will go bankrupt within three
years, and how should the model scores be turned into decisions when
bankruptcies are rare?

I picked this problem because it connects to my earlier capstone projects. In Project 1 I
built a cleaned dataset of Swiss legal entities from the GLEIF register, in
Project 2 I analyzed new company registrations from the Swiss commercial gazette. A
later capstone project will build a due diligence agent that looks up a
company and assesses it. Such an agent needs exactly the decision modeled
here. Given the numbers a company publishes, how urgently does a human
analyst need to look at it?

## Problem definition and dataset

**Task type:** supervised learning, binary classification. The observation unit is one company in one year, and the label says whether that company went bankrupt within the following three years.

The data covers Polish companies, collected from the Emerging Markets Information Service and donated to the UCI repository by Sebastian Tomczak. The bankrupt ones were observed from 2000 to 2012, the still operating ones from 2007 to 2013. It is described in Zieba, Tomczak and Tomczak (2016). The 64 feature columns are financial indicators from the annual statement, most of them ratios, plus two values that depend on company size, the logarithm of total assets (`Attr29`) and working capital (`Attr55`).

The archive holds five files, one per forecasting horizon. I use the `3year` file and leave the other four untouched. A risk screening wants warning ahead of time, not a confirmation shortly before the collapse. With a bankruptcy share under 5% this is an imbalanced classification problem.

I downloaded the archive on 2026-08-24 and keep it unchanged in the project folder.

## Setup

In [ ]:
import io
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

ARCHIVE = "polish+companies+bankruptcy+data.zip"
YEARS = 3
RANDOM_STATE = 42

## Load the dataset

I first count the rows of all five horizon files in the archive, then read the `3year` file and check that its size and class counts match the published description.

In [ ]:
def load_dataset(archive_path, years):
    """Load one horizon file of the UCI Polish bankruptcy archive.

    years picks the horizon, 1 to 5. An ARFF file is a CSV table with a
    small @attribute header, so the parsing is done by hand.
    """
    with zipfile.ZipFile(archive_path) as archive:
        with archive.open(f"{years}year.arff") as file:
            text = io.TextIOWrapper(file, encoding="utf-8").read()
    n_attributes = text.lower().count("@attribute") - 1
    columns = [f"Attr{i}" for i in range(1, n_attributes + 1)] + ["bankrupt"]
    data_section = text[text.lower().index("@data"):].split("\n", 1)[1]
    df = pd.read_csv(io.StringIO(data_section), header=None,
                     names=columns, na_values=["?"])
    df["bankrupt"] = df["bankrupt"].astype(int)
    return df

In [ ]:
# rows per horizon file, the file choice should be backed by numbers
for years in range(1, 6):
    rows = load_dataset(ARCHIVE, years).shape[0]
    marker = " (chosen)" if years == YEARS else ""
    print(f"{years}year file: {rows:,} companies{marker}")
print()

df = load_dataset(ARCHIVE, YEARS)

bankrupt_count = int(df["bankrupt"].sum())
print(f"Companies: {df.shape[0]:,}, columns: {df.shape[1]} "
      f"(64 financial indicators + target)")
print(f"Bankrupt within {YEARS} years: {bankrupt_count:,} "
      f"({df['bankrupt'].mean():.2%})")
print(f"Rows per bankruptcy: {df.shape[0] / bankrupt_count:.1f}")

The row counts add a second reason for the `3year` file, it is the largest of the five horizons, so training and evaluation get the most data. The chosen file also matches the published description: 10,503 rows with 64 financial indicators each. 495 of the rows carry the bankrupt label, roughly one in 21 (4.71%), a share that comes from how the file was built and is not a bankruptcy rate. The columns arrive under anonymous names `Attr1` to `Attr64`. I therefore keep the feature definitions from the UCI page in a table and look them up whenever a name needs interpreting.

In [ ]:
# feature definitions from the UCI dataset page, index = column name
FEATURE_DESCRIPTIONS = {
    "Attr1": "net profit / total assets",
    "Attr2": "total liabilities / total assets",
    "Attr3": "working capital / total assets",
    "Attr4": "current assets / short-term liabilities",
    "Attr5": "[(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365",
    "Attr6": "retained earnings / total assets",
    "Attr7": "EBIT / total assets",
    "Attr8": "book value of equity / total liabilities",
    "Attr9": "sales / total assets",
    "Attr10": "equity / total assets",
    "Attr11": "(gross profit + extraordinary items + financial expenses) / total assets",
    "Attr12": "gross profit / short-term liabilities",
    "Attr13": "(gross profit + depreciation) / sales",
    "Attr14": "(gross profit + interest) / total assets",
    "Attr15": "(total liabilities * 365) / (gross profit + depreciation)",
    "Attr16": "(gross profit + depreciation) / total liabilities",
    "Attr17": "total assets / total liabilities",
    "Attr18": "gross profit / total assets",
    "Attr19": "gross profit / sales",
    "Attr20": "(inventory * 365) / sales",
    "Attr21": "sales (n) / sales (n-1)",
    "Attr22": "profit on operating activities / total assets",
    "Attr23": "net profit / sales",
    "Attr24": "gross profit (in 3 years) / total assets",
    "Attr25": "(equity - share capital) / total assets",
    "Attr26": "(net profit + depreciation) / total liabilities",
    "Attr27": "profit on operating activities / financial expenses",
    "Attr28": "working capital / fixed assets",
    "Attr29": "logarithm of total assets",
    "Attr30": "(total liabilities - cash) / sales",
    "Attr31": "(gross profit + interest) / sales",
    "Attr32": "(current liabilities * 365) / cost of products sold",
    "Attr33": "operating expenses / short-term liabilities",
    "Attr34": "operating expenses / total liabilities",
    "Attr35": "profit on sales / total assets",
    "Attr36": "total sales / total assets",
    "Attr37": "(current assets - inventories) / long-term liabilities",
    "Attr38": "constant capital / total assets",
    "Attr39": "profit on sales / sales",
    "Attr40": "(current assets - inventory - receivables) / short-term liabilities",
    "Attr41": "total liabilities / ((profit on operating activities + depreciation) * (12 / 365))",
    "Attr42": "profit on operating activities / sales",
    "Attr43": "rotation receivables + inventory turnover in days",
    "Attr44": "(receivables * 365) / sales",
    "Attr45": "net profit / inventory",
    "Attr46": "(current assets - inventory) / short-term liabilities",
    "Attr47": "(inventory * 365) / cost of products sold",
    "Attr48": "EBITDA (profit on operating activities - depreciation) / total assets",
    "Attr49": "EBITDA (profit on operating activities - depreciation) / sales",
    "Attr50": "current assets / total liabilities",
    "Attr51": "short-term liabilities / total assets",
    "Attr52": "(short-term liabilities * 365) / cost of products sold",
    "Attr53": "equity / fixed assets",
    "Attr54": "constant capital / fixed assets",
    "Attr55": "working capital",
    "Attr56": "(sales - cost of products sold) / sales",
    "Attr57": "(current assets - inventory - short-term liabilities) / (sales - gross profit - depreciation)",
    "Attr58": "total costs / total sales",
    "Attr59": "long-term liabilities / equity",
    "Attr60": "sales / inventory",
    "Attr61": "sales / receivables",
    "Attr62": "(short-term liabilities * 365) / sales",
    "Attr63": "sales / short-term liabilities",
    "Attr64": "sales / fixed assets",
}


def describe_feature(name):
    """Return a readable label like "Attr1: net profit / total assets".

    Missing indicator columns from the imputer are mapped back to the
    feature they belong to.
    """
    if name.startswith("missingindicator_"):
        base = name.removeprefix("missingindicator_")
        return f"{base} is missing ({FEATURE_DESCRIPTIONS[base]})"
    return f"{name}: {FEATURE_DESCRIPTIONS[name]}"


pd.DataFrame({"what it measures": FEATURE_DESCRIPTIONS})

## Data checks

Before I model anything I want to see what a company row actually looks like, how complete the columns are, whether rows are duplicated and how the values are distributed.

In [ ]:
df.head()

Each row is one company observation. Most of the 64 feature columns are ratios covering profitability, leverage, liquidity and turnover, and `bankrupt` is the target. The column scales are very different. In these five rows `Attr1` stays below 0.19 and `Attr59` between 0 and 0.143, while `Attr62` runs from 65 to 102 and `Attr5` swings from -58.3 to 84.9. That spread alone tells me the logistic regression will need a scaler. One more thing stands out, all five rows carry label 0, and with a bankruptcy share under 5% that could easily be luck, so I check the row order instead of guessing.

In [ ]:
# the five head rows are all label 0, so I check whether the file is sorted by class
bankrupt_positions = np.flatnonzero(df["bankrupt"].to_numpy() == 1)
fifths = np.array_split(df["bankrupt"].to_numpy(), 5)

print(f"First and last row carrying the bankrupt label: "
      f"{bankrupt_positions[0]:,} and {bankrupt_positions[-1]:,}")
print(f"All bankrupt rows in one uninterrupted block: "
      f"{bool(np.all(np.diff(bankrupt_positions) == 1))}")
print("Bankruptcy share per fifth of the file: "
      + ", ".join(f"{part.mean():.2%}" for part in fifths))
print(f"Chance of five label-0 rows in a row by luck alone: "
      f"{(1 - df['bankrupt'].mean()) ** 5:.0%}")

The file is sorted by class. All 495 bankrupt rows sit in one uninterrupted block at the end, from row 10,008 to row 10,502, and the first four fifths of the file hold no bankruptcy at all. So the five zeros in `df.head()` proved nothing by themselves, luck alone produces them in 79% of cases at this bankruptcy share. The position check settles it. Every split from here on has to shuffle and stratify, because a split that cut the file by position would put all bankruptcies into one part.

In [ ]:
df.info()

All 64 indicators arrive as float columns and the target as an integer, so no type conversion is needed. The non-null counts differ from column to column. `Attr21` has 9,696 values, `Attr27` has 9,788, most others are complete or nearly complete. The "?" placeholder of the source file became a proper missing value because I passed it as `na_values` at load time.

In [ ]:
missing_share = (df.isna().mean() * 100).sort_values(ascending=False)
print(f"Columns with missing values: {(missing_share > 0).sum()} of {df.shape[1]}")
print(f"Rows with at least one missing value: {df.isna().any(axis=1).mean():.1%}")
missing_share.head(10).round(2)

The gaps are wider than the info output suggested. 44 of the 65 columns have missing values, and 53.5% of the rows have at least one. The feature definitions suggest that much of this missingness may be structural rather than accidental. `Attr37` (45.09% missing) divides by long-term liabilities, which is zero for a company without long-term debt. `Attr21` (7.68%) needs the previous year's sales, which a company observed for the first time does not have. `Attr27` (6.81%) divides by financial expenses, which can be zero. The dataset documentation does not state the cause, so I treat this as a plausible reading, not as a fact. Either way, a missing ratio may itself carry information about the company.

In [ ]:
duplicate_rows = df.duplicated()
in_duplicate_group = df.duplicated(keep=False)
feature_columns = list(df.columns[:-1])
label_conflicts = (df[df.duplicated(subset=feature_columns, keep=False)]
                   .groupby(feature_columns, dropna=False)["bankrupt"].nunique() > 1).sum()

print(f"Redundant duplicate rows (identical to an earlier row): {int(duplicate_rows.sum())}")
print(f"Rows that are part of a duplicate group: {int(in_duplicate_group.sum())}")
print(f"Feature-duplicate groups with conflicting labels: {int(label_conflicts)}")
print(f"Class of the redundant rows: "
      f"{df[duplicate_rows]['bankrupt'].value_counts().to_dict()}")

The file contains exact duplicates. 174 rows are part of a duplicate group, and 87 of them are redundant copies of an earlier row, 85 operating and 2 bankrupt. No duplicate group carries conflicting labels. So the copies do not hurt the label quality, but they are a problem for the evaluation. A random split can put one copy in training and an identical copy in validation or test, and the model would then be tested partly on rows it has already seen. The preparation therefore removes the redundant copies before any split.

In [ ]:
df.drop(columns="bankrupt").describe().T.round(3)

The quartiles look like ordinary balance sheet arithmetic. The median company earns a net profit of 4.3% of total assets (`Attr1`) and finances 46.4% of its assets with debt (`Attr2`). The extremes do not. `Attr5` ranges from minus 11.9 million to 685,440, with a standard deviation over 100,000, and several other ratios reach similar magnitudes. The likely mechanism is a denominator close to zero, for example a sales-based ratio for a company with almost no sales, but the data holds only the ratios and not their components, so I cannot verify it. I treat them as genuine but extreme values, and that matters only for the linear model. Tree splits use the order of the values, so the random forest is unaffected, while the logistic regression sees the ratios through a standard scaler and stays vulnerable to them.

In [ ]:
class_counts = df["bankrupt"].value_counts()
print(class_counts)
print(f"Bankruptcy share: {df['bankrupt'].mean():.4%}")

A model that never flags anything is already right about 95 of every 100 companies. That settles the metric question before a single model exists, accuracy cannot be the number I judge the models by. Four findings go into the preparation: the class imbalance, the 87 redundant duplicate rows, the widespread missing ratios, and the extreme values from near-zero denominators.

## Data preparation and preprocessing

I remove the redundant duplicate rows and split the data 60/20/20 into training, validation and test sets, stratified by the target. The remaining preprocessing runs inside the model pipelines of the next section and is therefore fitted on training data only.

In [ ]:
df_model = df.drop_duplicates().reset_index(drop=True)
print(f"Companies after removing redundant duplicates: {df_model.shape[0]:,} "
      f"(removed {df.shape[0] - df_model.shape[0]})")
print(f"Bankrupt: {int(df_model['bankrupt'].sum())} "
      f"({df_model['bankrupt'].mean():.4%})")

X = df_model.drop(columns="bankrupt")
y = df_model["bankrupt"]

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=RANDOM_STATE)

for name, X_part, y_part in [("train", X_train, y_train),
                             ("validation", X_val, y_val),
                             ("test", X_test, y_test)]:
    print(f"{name}: {X_part.shape[0]:,} companies x {X_part.shape[1]} features, "
          f"bankrupt = {int(y_part.sum()):,} ({y_part.mean():.4%})")

**Why this preparation was needed:** the duplicates would have made the evaluation look better than it is, and an unstratified split would have put different bankruptcy shares into the three parts. After removing the 87 redundant copies, 10,416 unique company records remain, 493 of them bankrupt (4.73%). The split keeps the bankruptcy share close to the overall 4.73%, at 4.74% in training, 4.70% in validation and 4.75% in test, so comparisons across the parts are fair. The validation set carries the model comparison and the threshold selection. The test set is touched exactly once, in the final evaluation at the end. One note on wording, the file has no company identifiers, so I say company records. Whether two records describe the same underlying company is not verifiable.

## Model selection and training

I train four models: a dummy baseline that always predicts "not bankrupt", a hand-written solvency rule, a logistic regression and a random forest with 300 trees. All model pipelines impute missing values with the median and add a missing indicator column per affected feature. For the dummy this changes nothing, it never looks at the features, the imputer only keeps the pipelines uniform. The logistic regression additionally scales the features. Every configuration is fixed in advance, I run no hyperparameter search. I fit the models one after the other and note after each one what its output tells me.

In [ ]:
# baseline 1, always predicts the majority class "not bankrupt"
dummy_model = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", DummyClassifier(strategy="most_frequent")),
])

start = time.time()
dummy_model.fit(X_train, y_train)
print(f"Dummy baseline fitted in {time.time() - start:.1f} seconds")

train_pred_dummy = dummy_model.predict(X_train)
print(f"Distinct predicted classes on the training set: {np.unique(train_pred_dummy).tolist()}")
print(f"Training accuracy: {accuracy_score(y_train, train_pred_dummy):.4f}")

The dummy needs no real training. It stores which class is the more frequent one and predicts it for every company. The check confirms it, the only predicted class is 0, and that is right for 95.26% of the 6,249 training companies. That is the anchor every accuracy value below has to be read against.

In [ ]:
def rule_predict(X):
    """Apply the deterministic solvency rule to a feature matrix.

    Returns 1 per company when the net profit is negative and more than
    half of the assets are financed by debt. Missing ratios count as no
    warning sign.
    """
    loss_making = X["Attr1"].fillna(0) < 0
    highly_leveraged = X["Attr2"].fillna(0) > 0.5
    return (loss_making & highly_leveraged).astype(int).to_numpy()

In [ ]:
# baseline 2 needs no training, I only check how often it flags at all
print(f"Rule flags on the training set: {int(rule_predict(X_train).sum()):,} "
      f"of {len(X_train):,} companies")

The rule needs no training either. It applies two fixed thresholds, so I only check how often it flags at all. The two ratios are the classic loss-making and highly-leveraged warning signs, and I picked them before looking at any model output. On the training set the rule flags 926 of the 6,249 companies. That is about three times as many as the 296 that actually went bankrupt, so it is a broad filter, not a precise one.

In [ ]:
# model 1, logistic regression with scaling inside the pipeline
logreg_model = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=5000,
                                 random_state=RANDOM_STATE)),
])

start = time.time()
logreg_model.fit(X_train, y_train)
print(f"Logistic regression fitted in {time.time() - start:.1f} seconds")

The logistic regression fits in a tenth of a second. It is in the comparison for three reasons: it is interpretable, it outputs the probabilities that a threshold analysis needs, and it connects to the linear models from my coursework. The balanced class weights matter here. Without them the model would likely optimize mostly for the 95.3% majority class and predict very few bankruptcies. This is the textbook expectation for imbalanced data, I did not test the unweighted variant.

In [ ]:
# model 2, random forest, no scaling needed for trees
forest_model = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", RandomForestClassifier(n_estimators=300,
                                     class_weight="balanced_subsample",
                                     random_state=RANDOM_STATE, n_jobs=-1)),
])

start = time.time()
forest_model.fit(X_train, y_train)
print(f"Random forest fitted in {time.time() - start:.1f} seconds")

The random forest takes 0.7 seconds, so training cost plays no role anywhere in this project. It is in the comparison because it captures interactions between ratios that a linear model cannot represent, and because it reports feature importances I can inspect afterwards. Like the logistic regression it runs with balanced class weights, here in the per-tree `balanced_subsample` variant. All four models are now in place, and none of them has seen the validation or the test companies.